PROGETTO 3: ANALISI DEL SENTIMENT SU RECENSIONI DI FILM

Passiamo dal computer vision al Natural Lenguage Processing (NLP)
In questo progetto l'input è una frase, l'output probabilità che la recensione sia positiva o negativa. E' quindi una classificazione binaria supervisionata.
0=negativa 1=positiva
La difficoltà non è la classificazione finale. E' trasformare il linguaggio umano in numeri senza distruggere l'ordnie, significato e contesto.

Capire l'emozioni umane attraverso il testo.
Immagina di trovarti davanti ad un oceano di milioni di commenti, per un essere umano imossibile da leggere tutti.
Costruiamo un setaccio capace di separare l'entusiamo dalla frustazine. Trasformando parole confuse in dati strategici.

Non utiliziamo i soliti esempi da manule, usiamo i dati del mondo reale

- Gestione di dataset reali e rumorosi
- Banchmarking architetture: confronto prestazionale e struttura gra GRU e LSTM
- Diagnostica del modello: autopsia degli errori del modello, perchè una macchina non capisce il sarcasmo

Dataset Amazon Reviews
La complessità del linguaggio naturale nel commercio
A differenza dei dataset accademici puliti, le recensioni di Amazon presentano una varianza estreme in termini di lunghezza, grammatica e registro linguistico.
La sfida principale risiede nel trasformare questo rumore in un segnale strutturato.
L'analisi del sentiment su larga scala richiede una pipeline di pre-elaborazione che bilanci la richezza semantica con l'efficienza computazionale necessaria per milioni di campioni.

E come prepariamo questa catena di montaggio per i dati?

Preprocessing per Grandi Scale
Dall'universo di parole al vocabolario ottimizzato
- Tokenizzazione Subword: utilizzo di algoritmi per gestire parole rare o errori di battitura senza esplosione del vocabolario. Permetti di capire che eccezionale e ecezinale, condividono la stessa radice semantica.
- Bilanciamento delle Classi: tecniche di campionamento per gestire la prevalenza tipica di recensioni estremamente positive o negative. Dobbiamo bilanciare le classi per evitare che il modelo diventi un ottimista o un pessimista.
- Troncamento Dinamico: scelta della lunghezza massima della sequenza basata sulla distribuzione statistica dei testi per non perdere contesto critico.
- Embedding Pre-addestrati: integrazione di pesi iniziali derivanti da modelli linguistici globali per accellelare la convergenza su domini specifici. E' come dare al modello un dizionario già scritto invece di fargli inventare la lingua da zero.

Una volta pulite le parole dobbiamo trasformare le parole in numeri che la matematica può masticare.

Vettorizzazione e Sparsità
Il TF-IDF che come contare quanta è rara una parola in un libro, è utile ma cieco al contesto. 
I modelli neurali preferiscono gli embedding densi, immagina di mappare una parola in una nota 3d, buono ed ottimosaranno vicini, mentre terribile sarà dall'altra parte. Useremo anche masking affinche i calcoli ignorino i valori nulli aggiunti per uniformare le lunghezze.
I termini 'Out of Vocabulary' vengono mappati su un token speciale per mantenere l'integrità della sequenza durante l'inferenza.

Come portiamo milioni di recensioni dentro la nostra GPU senza farla esplodere?
Caricare tutto in memoria sarebbe un errore da pricnipianti. Sasrebbe come cercare di svuotare un intero silos di grano in una pentola. Useremo il generatore di dati e la pipeline tf.data. Come un nastro trasportarore che prepara il prossimo batch di recensioni mentre la GPU sta ancora analizzando il precedente. Questo ci permette di lavorare su dataset enormi.

Adesso che i dati scorrono è il momento di scegliere il motore, GRU o LSTM?

Duello Ricorrente: GRU vs LSTM
Scegliere il motore della memoria
LSTM e GRU sono le colonne portanti delle reti ricorrenti moderne. Sebbene simili, la loro struttura interna offre vantaggi diversi in termini di capacità di memoria e velocità di calcolo
Entrambe progettate per avere memoria a lungo termine.
Analiziamo i trade-off tra la complessità della cella LSTM e la snellezza operativa della GRU nel contesto delle recensioni testuali.

Le LSTM meticolosa e complessa, l GRU più semplice

Differenza Architetturali.
Meccanismi di controllo del flusso
- Forget Gate LSTM: capacità di rimuovere esplicitamente informazioni irrilevanti dal 'cell.state' per lunghi periodi
- Update Gate GRU: fusione dei gate di input e forget in un unico meccanismo che decide quanta parte dello stato precedente mantenere
- Cell State vs Hidden State: la LSTM mantiene un'autostrada di informazione separata, mentre la GRU lavora esclusivamente con lo stato nascosto.
- Complessita Computazionale: la GRU possiede circa il  25% in meno di parametri, risultando significativamente più veloce durante il training.

Ma la velocità della GRU sacrifica la qualità della memoria?

Dinamiche di Apprendimento
La GRU tende a convergere più rapidamente su dataset di medie dimensioni grazie al minor numero di operazioni per passo temporale.
Grazie al gate di aggiornamento può decidere di far passare un informazione imnportante attraverso molti layer senza modificarla. questo la rende molto scattante in termini di convergenza.
Tuttavia, per testi estremamemnte lunghi, la LSTM mantiene ancora un leggero vantaggio di stabilità, può offire una stabilità superiore grazie alla gestione granulare del 'forgetting'.
L'integrazione del gate di aggiornamento permette alla rete di bypassare layer ricorrenti se il segnale corrente non è informativo

Vediamo i numeri dietro a questa efficienza

Analisi dei Parametri
Efficienza e Overfitting
Con meno parametri, la GRU, essendo più snella, agisce come una forma di regolarizzazione implicita, essendo meno incline a memorizzare il rumore rispetto a una LTSM sovradimensionata.
Tuttavia, se il dataset è 'infinito', la maggiore capacità della LSTM permete di estrarre pattern più complessi e sottili

Ma anche il miglior medello del mondo ogni tanto può sbagliare.

Analisi dei Casi di Errore
Diagnostica oltre l'accuratezza
L'accuratezza globale spesso nasconde debolezze sistemiche del modello. Per un deploy responsabile, dobbiamo capire 'perchè' la rete fallisce su determinate recensioni.
Esaminiamo le discrepanza tra sentiment reale e predetto, focalizzandoci su fenomeni linguistici che mettono in crisi le reti ricorrenti.

Tipologia di Fallimento
Limiti semantici e sintattici.
- Sarcasmo e Ironia: frasi con parole positive che intendono un significato negativo, difficili da catturare senza una comprensione profonda del contesto. Nemico numero 1 per le nostre reti.
- Negazione e Lungo Raggio: strutture dove la particella negativa è molto distante dal termine che modifica, mettendo alla prova la memoria della rete.
- Recensioni Contrastanti: testi che contengono sia elogi che critiche feroci, dove il modello fatica a pesare la rilevana delle diverse parti.
- Ambiguità Lessicale: parole che cambiano significato in base alla categoria del prodotto (es. 'freddo' per un gelato vs un riscaldatore)

Ma come faccimao a diagnosticare questi fallimenti?

Strumenti di Verifica
* Matrice di confusione
    La matrice di confusione permette di vedere se il modello è sbagliato (bias) verso recensioni positive o se confonde gradi intermedi di neutralità
* Saliency Maps
    Visualizzazione dei gradienti per identificare quali parole specifiche hanno attivato maggiormente la decisione finale del modello
* F1-score bilanciato
    Fornisce una misura della precisione e del recupero che non viene distorta dallo sbilanciamento del dataset originale.

Ma un modello che funziona per i libri, funziona anche per frigoriferi?

Validazione Incrociata dei Domini
Generalizzazione esterna
Un modello addestrato su recensioni di libri potrebbe fallire su recensioni di elettronica. Questa degradazione indica un overfitting sulle feature specifiche del dominio.
L'analisi dei residui e degli errori sistematici guida la scelta tra aggiungere dati o cambiare l'architettura verso approci più flessibili come l'attenzione.

In [2]:
# %pip install --upgrade datasets
import os

# 1. SETUP DEL BACKEND (Best Practice 2026)
# Keras 3 permette di scrivere codice una volta ed eseguirlo ovunque. 
# Forziamo PyTorch come motore matematico per sfruttare i suoi kernel ottimizzati.
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import torch
import pandas as pd
from keras import layers, ops
# Libreria standard 2026 per l'accesso immediato ai dati
from datasets import load_dataset 

# ==========================================
# 2. CARICAMENTO DATASET REALE
# ==========================================
print("[INFO] Caricamento recensioni da Amazon Polarity...")

try:
    # Identificativo completo: namespace/nome_dataset
    streaming_dataset = load_dataset(
        "fancyzhx/amazon_polarity",
        split="train",
        streaming=True
    )

    # Mescolamento approssimato tramite buffer.
    # Evita di prendere necessariamente solo le prime recensioni.
    streaming_dataset = streaming_dataset.shuffle(
        seed=42,
        buffer_size=10_000
    )

    # In streaming vengono scaricati soltanto i record necessari.
    rows = list(streaming_dataset.take(5000))

    if not rows:
        raise ValueError("Il dataset caricato è vuoto.")

    # Utilizziamo sia il titolo sia il contenuto della recensione.
    X_raw = [
        f"{row.get('title', '')} {row.get('content', '')}".strip()
        for row in rows
    ]

    y_raw = [row["label"] for row in rows]

    if len(X_raw) != len(y_raw):
        raise ValueError(
            f"Numero testi diverso dal numero etichette: "
            f"{len(X_raw)} testi, {len(y_raw)} etichette."
        )

    print(
        f"[INFO] Dataset pronto: "
        f"{len(X_raw)} campioni caricati con successo."
    )

except Exception as e:
    # Non proseguire con tre frasi inventate:
    # il training risultante sarebbe privo di valore.
    raise RuntimeError(
        f"Impossibile caricare Amazon Polarity: {e}"
    ) from e


# ==========================================
# 3. PREPROCESSING PIPELINE
# ==========================================
MAX_VOCAB_SIZE = 10000 
MAX_SEQUENCE_LENGTH = 150 

vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH,
)

# [TEORIA] L'adattamento deve avvenire su un array di stringhe non vuoto.
# Assicuriamoci che ogni elemento sia una stringa (evita errori di tipo Float/None).
X_raw_clean = [str(text) for text in X_raw]

print("[INFO] Adattamento TextVectorization in corso...")
vectorizer.adapt(np.array(X_raw_clean))

# 1. Trasformazione in sequenze numeriche
X_seq = vectorizer(np.array(X_raw_clean))
y_seq = np.array(y_raw).astype("float32")

# 2. Risoluzione errore Device (GPU -> CPU)
# Portiamo i dati sulla CPU per poterli manipolare con NumPy
X_seq_numpy = X_seq.cpu().numpy() if hasattr(X_seq, "cpu") else X_seq.numpy()

# 3. DEFINIZIONE DEGLI INDICI (Risoluzione NameError)
# Teoria: Creiamo un array di numeri da 0 a N-1 e lo mescoliamo casualmente.
# Questo garantisce che X e y rimangano sincronizzati durante lo shuffling.
indices = np.arange(len(X_seq_numpy))
np.random.shuffle(indices)

# 4. Applicazione dello shuffling
X_seq_shuffled = X_seq_numpy[indices]
y_seq_shuffled = y_seq[indices]

# 5. Split Training/Test (80/20)
split_idx = int(len(X_seq_shuffled) * 0.8)
X_train, X_test = X_seq_shuffled[:split_idx], X_seq_shuffled[split_idx:]
y_train, y_test = y_seq_shuffled[:split_idx], y_seq_shuffled[split_idx:]

print(f"[INFO] Shuffle completato. Training set: {len(X_train)}, Test set: {len(X_test)}")

# ==========================================
# 4. DEFINIZIONE DEL MODELLO (LSTM vs GRU)
# ==========================================
# Teoria: Usiamo mask_zero=True. Questo dice alla RNN di ignorare i padding (0) 
# evitando di degradare il segnale del gradiente su frasi corte.
def build_sentiment_model(cell_type="lstm"):
    model = keras.Sequential([
        layers.Input(shape=(MAX_SEQUENCE_LENGTH,)),
        # Embedding: crea rappresentazioni vettoriali in uno spazio semantico a 128 dimensioni.
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, mask_zero=True),
        
        # Scelta dell'architettura: la Bidirectional LSTM legge in entrambi i sensi,
        # fondamentale per capire negazioni poste a fine frase.
        layers.Bidirectional(layers.LSTM(64)) if cell_type == "lstm" else layers.Bidirectional(layers.GRU(64)),
        
        layers.Dropout(0.4), # Dropout aumentato per gestire il rumore dei dati Amazon.
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid') # Probabilità binaria finale.
    ])
    
    # AdamW (Weight Decay) per una regolarizzazione superiore su dati testuali complessi.
    model.compile(optimizer=keras.optimizers.AdamW(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return model

# ==========================================
# 5. BENCHMARKING E TRAINING
# ==========================================
print("\n--- Training LSTM Model (Amazon Real Data) ---")
lstm_model = build_sentiment_model("lstm")
# Usiamo EarlyStopping per evitare overfitting se la loss di validazione smette di scendere.
lstm_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=64)

print("\n--- Training GRU Model (Amazon Real Data) ---")
gru_model = build_sentiment_model("gru")
gru_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=64)

# ==========================================
# 6. SALVATAGGIO "BEST PRACTICE 2026"
# ==========================================
# Il formato .keras v3 include l'intera pipeline di preprocessing TextVectorization.
lstm_model.save("amazon_sentiment_pro_2026.keras")
print("\n[INFO] Modello salvato nel formato universale V3.")

# ==========================================
# 7. ANALISI DEI CASI DI ERRORE
# ==========================================
def analyze_errors(model, texts, labels):
    # Applichiamo la vettorizzazione in tempo reale per l'inferenza
    processed_input = vectorizer(np.array(texts))
    preds = model.predict(processed_input, verbose=0)
    preds_binary = (preds > 0.5).astype(int).flatten()
    
    print("\n--- ANALISI QUALITATIVA SUI DATI REALI ---")
    for i in range(len(texts)):
        status = "CORRETTO" if preds_binary[i] == labels[i] else "ERRORE"
        print(f"[{status}] Testo: {texts[i][:80]}...")
        print(f"Target: {labels[i]} | Pred: {preds_binary[i]} (Confidenza: {preds[i][0]:.2f})\n")

# Test su frasi nuove che contengono sfide semantiche (Negazioni e Sarcasmo)
real_world_tests = [
    "I expected much more from this software, a huge disappointment.", # Negativo
    "Surprisingly good, it actually solved all my bugs!",              # Positivo con parola 'bugs' (spesso negativa)
    "Great, another update that breaks everything. Thanks Amazon."    # Sarcasmo (Difficile)
]
test_labels = [0, 1, 0]

analyze_errors(lstm_model, real_world_tests, test_labels)

[INFO] Caricamento recensioni da Amazon Polarity...


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barbara\.cache\huggingface\hub\datasets--fancyzhx--amazon_polarity. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[INFO] Dataset pronto: 5000 campioni caricati con successo.
[INFO] Adattamento TextVectorization in corso...
[INFO] Shuffle completato. Training set: 4000, Test set: 1000

--- Training LSTM Model (Amazon Real Data) ---
Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 60s 958ms/step - accuracy: 0.6817 - loss: 0.6017 - val_accuracy: 0.8100 - val_loss: 0.4319
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 63s 997ms/step - accuracy: 0.9075 - loss: 0.2458 - val_accuracy: 0.8770 - val_loss: 0.2968
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.9765 - loss: 0.0767 - val_accuracy: 0.8590 - val_loss: 0.4018
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.9937 - loss: 0.0249 - val_accuracy: 0.8610 - val_loss: 0.3844
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 62s 983ms/step - accuracy: 0.9967 - loss: 0.0128 - val_accuracy: 0.8490 - val_loss: 0.6928

--- Training GRU Model (Amazon Real Data) ---
Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.7163 - loss: 0.5901 - val_accuracy: 0